[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anicka-net/nla-at-home/blob/main/notebooks/03_roundtrip_faithfulness.ipynb)

# 03 · Round-Trip & Faithfulness

### HAAISS workshop — hands-on part 3 of 3 (code-along)

So far we went **activation → English** (the AV, verbalizer). There is a second adapter that goes back **English → activation** (the AR, *reconstructor*). Chaining them gives a **round-trip**:

```
vector  --AV-->  caption  --AR-->  vector'
```
If the caption really captured the vector, `vector'` should point the same way as `vector` (**high cosine**). If the caption hallucinated, the cosine drops — so cosine becomes a **faithfulness detector**. That's the whole idea behind the compass / gap metric we use to catch a lying NLA.

## Setup — same base, two adapters
The clever part: **AV and AR are both LoRA adapters on the same Qwen base.** We load the base once, attach both, and hot-swap. That's why the round-trip fits a free T4.

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE       = "Qwen/Qwen2.5-7B-Instruct"          # the model whose mind we read
AV_ADAPTER = "anicka/nla-qwen2.5-7b-L20-av-v2"    # the "verbalizer" (activation -> English)
LAYER      = 20                                   # single-layer NLA lives at Qwen layer 20
DEPTH_PCT  = 71                                   # <-- NOT cosmetic. The adapter was TRAINED
                                                  #     at 71% depth (layer 20 of 28). This
                                                  #     number is a CONDITIONING INPUT to the
                                                  #     verbalizer. Notebook 02 lets you feel
                                                  #     what happens when you lie about it.
INJECT_CHAR  = "\u320e"                          # the placeholder token we overwrite: ㈎
INJECT_SCALE = 150.0                              # we normalize the activation's L2 norm TO this

In [ ]:
AR_ADAPTER = "anicka/nla-qwen2.5-7b-L20-ar-v2"   # English -> activation

In [ ]:
device = "cuda"
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"

# 4-bit so a 7B model + adapters fit a free-Colab T4 (16 GB). fp16 compute:
# the GRPO-sharpened adapter is numerically sensitive, and fp16 on CUDA is a
# tested-safe path (bf16 on Apple MPS collapses it; not our case here).
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)

tok  = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb,
                                            device_map={"": 0})
model = PeftModel.from_pretrained(base, AV_ADAPTER).eval()   # adapter name = "default"

inject_id = tok.encode(INJECT_CHAR, add_special_tokens=False)
assert len(inject_id) == 1, f"injection char must be ONE token, got {inject_id}"
inject_id = inject_id[0]
print("loaded — base + AV adapter on", next(model.parameters()).device)

In [ ]:
def get_layers(m):
    """Reach the transformer block list through the PEFT + CausalLM wrappers."""
    b = m.base_model.model if hasattr(m, "base_model") else m
    inner = b.model if hasattr(b, "model") else b
    return inner.layers

def read_activation(prompt, layer=LAYER, max_new_tokens=128):
    """Run the model on `prompt` and grab the residual-stream vector at `layer`,
    at the last prompt token (the position that decides the next word)."""
    chat = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                   tokenize=False, add_generation_prompt=True)
    inp = tok(chat, return_tensors="pt").to(device)

    grab = {}
    def hook(mod, inpt, out):
        h = out[0] if isinstance(out, tuple) else out
        if "h" not in grab:                 # FIRST forward pass only — otherwise
            grab["h"] = h[:, -1, :].detach() # every generated token overwrites it
    handle = get_layers(model)[layer].register_forward_hook(hook)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    handle.remove()
    reply = tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
    return grab["h"].squeeze(0), reply

def normalize_to(v, scale=INJECT_SCALE):
    """Rescale v so its L2 norm equals `scale`. NOT v * scale — see notebook 02."""
    n = v.float().norm().clamp_min(1e-12)
    return v * (scale / n)

def av_prompt(depth_pct):
    return (
        "You are a meticulous AI researcher conducting an important investigation "
        "into activation vectors from a language model. Your overall task is to "
        "describe the semantic content of that activation vector.\n\n"
        "We will pass the vector enclosed in <concept> tags into your context, "
        "along with the network depth where it was extracted. "
        "You must then produce an explanation for the vector, enclosed within "
        "<explanation> tags. The explanation consists of 2-3 text snippets "
        "describing that vector.\n\n"
        f"Here is the vector from depth {depth_pct}% of the network:\n\n"
        f"<concept>{INJECT_CHAR}</concept>\n\n"
        "Please provide an explanation.\n\n"
        "<explanation>")

def describe(activation, depth=DEPTH_PCT, max_new_tokens=120, scale_fn=normalize_to):
    """The whole NLA read: build the prompt, overwrite the placeholder token's
    embedding with the (rescaled) activation, let the model narrate."""
    ids = tok.encode(av_prompt(depth), add_special_tokens=True)
    pos = ids.index(inject_id)
    emb = model.get_input_embeddings()(torch.tensor([ids], device=device)).clone()
    emb[0, pos, :] = scale_fn(activation.to(emb.dtype))
    with torch.no_grad():
        out = model.generate(inputs_embeds=emb, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id)
    seq = out[0]
    gen = seq[len(ids):] if seq.shape[0] > len(ids) else seq  # embeds path returns new-only
    return tok.decode(gen, skip_special_tokens=True).split("</explanation>")[0].strip()

In [ ]:
# attach the reconstructor alongside the verbalizer on the SAME base model
model.load_adapter(AR_ADAPTER, adapter_name="ar")   # AV is "default"
print("adapters:", list(model.peft_config.keys()))

## The reconstructor

The AR adapter is trained so that, when it reads a caption, the residual stream at **layer 20** *becomes* the activation being described. No extra head: the reconstruction is literally the hidden state at layer 20, last token.

In [ ]:
import torch.nn.functional as F
AR_TEMPLATE = (
    "You are a meticulous AI researcher conducting an important investigation "
    "into a model's internal states. Below is a description of an activation "
    "vector:\n\n<explanation>{explanation}</explanation>\n\n"
    "Based on this description, reconstruct the activation vector.")

def reconstruct(description):
    model.set_adapter("ar")
    ids = tok.encode(AR_TEMPLATE.format(explanation=description),
                     add_special_tokens=True)
    with torch.no_grad():
        out = model(input_ids=torch.tensor([ids], device=device),
                    output_hidden_states=True, use_cache=False)
    model.set_adapter("default")                    # switch back to the AV
    return out.hidden_states[LAYER + 1][0, -1].float().cpu()

def cos(a, b):
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

## The round-trip

In [ ]:
prompt = "Explain how a hash map handles collisions."
model.set_adapter("default")
# Capture the activation from the CLEAN BASE model (adapter off): the AR
# was trained on base-model activations, and "reading the model's mind"
# should mean the model's mind — not the verbalizer's.
with model.disable_adapter():
    activation, reply = read_activation(prompt)
caption = describe(activation)
back    = reconstruct(caption)

print("caption      :", caption)
print("round-trip cos:", round(cos(activation.float().cpu(), back), 3))

> **Anchor:** a full round-trip (NLA's *own* caption → AR) lands around **0.5–0.6** cosine on this model — that is the real, honest number, not the 0.94 you'd get feeding the AR a hand-written human caption. Round-tripping through an imperfect verbalizer is lossy; that gap is exactly the research problem.

## Faithfulness as a gap — and a trap

Now the payoff. Take the *real* caption and a deliberately *wrong* one, reconstruct both, and compare cosine to the true activation. First, the **naive** way — watch it fail:

In [ ]:
wrong = "- Recipe for Thai green curry with coconut milk and basil\n"\
        "- Step-by-step cooking instructions for dinner"

c_real  = cos(activation.float().cpu(), reconstruct(caption))
c_wrong = cos(activation.float().cpu(), reconstruct(wrong))
print(f"cos(real caption)  = {c_real:.3f}")
print(f"cos(wrong caption) = {c_wrong:.3f}")
print(f"naive gap = {c_real - c_wrong:+.3f}   ...both ~0.6 and the gap is noise. Why?")

Both cosines land around **0.6** and the gap is a coin-flip (±0.01–0.03 — measured on the exact T4 you are using). The reason: **every** AR reconstruction shares one huge component — the mean of the reconstruction distribution. Raw cosine mostly measures that shared mean, and the actual *content* signal lives in the small deviation from it. (Notebook 04 builds this same lesson for raw activations; here it bites the reconstructions.) So we **center**: reconstruct a handful of distractor captions, subtract their mean, and compare in deviation space.

In [ ]:
# === EDIT THE DISTRACTORS to probe the detector ===
DISTRACTORS = [
    wrong,  # the curry recipe from above
    "- Legal contract clause about liability limitation active\n- Formal register, defined terms",
    "- Football match commentary, goal celebration active\n- Present-tense excited sports narration",
    "- Romantic poetry about moonlight and longing\n- Metaphor-dense lyrical register",
    "- Python exception traceback analysis active\n- Debugging context, error-message vocabulary",
]
recons = [reconstruct(d) for d in DISTRACTORS]
mean_recon = torch.stack(recons).mean(0)

a_dev    = activation.float().cpu() - mean_recon
true_dev = reconstruct(caption) - mean_recon
scores   = {"TRUE caption": cos(a_dev, true_dev)}
for d, r in zip(DISTRACTORS, recons):
    scores[d.split(chr(10))[0][:48]] = cos(a_dev, r - mean_recon)

ranked = sorted(scores.items(), key=lambda kv: -kv[1])
for name, s in ranked:
    mark = " <-- the vector votes for this one" if name == "TRUE caption" else ""
    print(f"{s:+.3f}  {name}{mark}")

centered_gap = scores["TRUE caption"] - max(v for k, v in scores.items() if k != "TRUE caption")
print(f"\ncentered gap = {centered_gap:+.3f}   (positive => the vector prefers the truth)")

Try harder distractors — a *near-miss* (same domain, wrong detail) vs a *wild miss* (unrelated topic). The centered gap should shrink for near-misses: the detector is graded, not binary. That gradient is what makes the compass metric usable as a training signal (rerank candidate captions by centered reconstruction cosine, keep the faithful ones).

---
### ✅ Self-check
Expected: both adapters load; the **naive** gap is ~0 (that is the lesson, not a bug); the **TRUE caption ranks #1** in the centered comparison with a clearly positive centered gap (≈ +0.1–0.25 measured on a T4). If the true caption does not win, something is off — check `LAYER+1` indexing and that `set_adapter` actually switched (`model.active_adapter`).

In [ ]:
assert ranked[0][0] == "TRUE caption", "centered ranking failed — see self-check note"
assert centered_gap > 0.03, f"centered gap suspiciously small: {centered_gap:+.3f}"
print(f"self-check: TRUE caption ranks #1, centered gap {centered_gap:+.3f} ✓")